# 03 - Ticket groups + models for the leftover passengers

**Starting point:** notebook 02's family rule scored **0.799** on Kaggle.

Two ideas tried here:
1. **Ticket groups:** maids, nannies and friends often shared a family's ticket but had a different surname. The family rule missed them.
2. **A model for passengers with no family group:** men, and women travelling alone. Can a model beat "women live, men die" for them?

**Result in short:** idea 1 changes **1** test passenger. Idea 2 does **not** help, so it's left out.
Output: `submission_v3.csv`.

In [1]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score

train = pd.read_csv("titanic/train.csv")
test = pd.read_csv("titanic/test.csv")
df = pd.concat([train, test], ignore_index=True)
is_train = df.Survived.notna()

# Same family groups as notebook 02
df["Title"] = df.Name.str.extract(r",\s*([^\.]+)\.")[0]
df["IsWomanOrChild"] = (df.Sex == "female") | (df.Title == "Master")
df["Surname"] = df.Name.str.split(",").str[0]
df["GroupId"] = (df.Surname + "-" + df.Pclass.astype(str) + "-" + df.Ticket.str[:-1]
                 + "-" + df.Fare.astype(str) + "-" + df.Embarked.fillna(""))
df.loc[~df.IsWomanOrChild, "GroupId"] = "none"
df.loc[df.groupby("GroupId").GroupId.transform("size") < 2, "GroupId"] = "none"
print("Women/children in a family group:", (df.GroupId != "none").sum())

Women/children in a family group: 219


## Idea 1: Ticket groups
A woman or boy with no family group joins a group if they share its **exact ticket number**.

In [2]:
no_group = df.index[(df.GroupId == "none") & df.IsWomanOrChild]
joined = []
for i in no_group:
    match = df.loc[(df.GroupId != "none") & (df.Ticket == df.at[i, "Ticket"]), "GroupId"]
    if len(match):
        df.at[i, "GroupId"] = match.iloc[0]
        joined.append(i)

print("Joined a group by ticket:", len(joined))
df.loc[joined, ["PassengerId", "Name", "Ticket", "Survived"]].assign(InTest=~is_train[joined])

Joined a group by ticket: 9


,PassengerId,Name,Ticket,Survived,InTest
337,338,"Burns, Miss. Elizabeth Margaret",16966,1.0,False
708,709,"Cleaver, Miss. Alice",113781,1.0,False
950,951,"Chaudanson, Miss. Victorine",PC 17608,NaN,True
1032,1033,"Daniels, Miss. Sarah",113781,NaN,True
1067,1068,"Sincock, Miss. Maude",C.A. 33112,NaN,True
1258,1259,"Riihivouri, Miss. Susanna Juhantytar Sanni""""",3101295,NaN,True
1262,1263,"Wilson, Miss. Helen Alice",16966,NaN,True
1266,1267,"Bowen, Miss. Grace Scott",PC 17608,NaN,True
1291,1292,"Bonnell, Miss. Caroline",36928,NaN,True


## The family rule (same as notebook 02)

In [3]:
def predict(rows, labelled, leave_one_out=False):
    stats = labelled[labelled.GroupId != "none"].groupby("GroupId").Survived.agg(["sum", "count"])
    pred = (rows.Sex == "female").astype(int)
    for i, r in rows.iterrows():
        if r.GroupId not in stats.index:
            continue
        s, c = stats.loc[r.GroupId]
        if leave_one_out:
            s, c = s - r.Survived, c - 1
        if c == 0:
            continue
        rate = s / c
        if r.Title == "Master" and rate == 1:
            pred[i] = 1
        elif r.Sex == "female" and rate == 0:
            pred[i] = 0
    return pred

trn, tst = df[is_train], df[~is_train]
print(f"Train check (leave-one-out): {(predict(trn, trn, leave_one_out=True) == trn.Survived).mean():.3f}  (same as v2)")

Train check (leave-one-out): 0.846  (same as v2)

### What changes on test?
The train check is the same, because most newly joined passengers are in test.
So compare directly with the v2 file.

In [4]:
pred = predict(tst, trn)
v2 = pd.read_csv("submission_v2_group.csv")
changed = pred.values != v2.Survived.values
print("Test passengers changed vs v2:", changed.sum())
tst[changed][["PassengerId", "Name", "Ticket"]].assign(v2=v2.Survived.values[changed], v3=pred.values[changed])

Test passengers changed vs v2: 1


,PassengerId,Name,Ticket,v2,v3
1258,1259,"Riihivouri, Miss. Susanna Juhantytar Sanni""""",3101295,1,0


**Why the change makes sense:** Miss Riihivouri shared ticket 3101295 with the Panula family.
All 6 Panulas in train died, including the mother and three boys. So she is now predicted to die.

## Idea 2: A model for passengers with no family group
For these passengers the rule just uses "women live, men die".
Can a model do better? The table compares 3 models with that simple guess (5-fold cross-validation).

Features: class, family size, people on the ticket, fare per person, age, boarding port.

In [5]:
df["FamilySize"] = df.SibSp + df.Parch + 1
df["TicketCount"] = df.groupby("Ticket").Ticket.transform("size")
df["FarePerPerson"] = df.Fare.fillna(df.Fare.median()) / df.TicketCount
df["AgeFilled"] = df.Age.fillna(-1)
df["FromS"] = (df.Embarked == "S").astype(int)
df["FromC"] = (df.Embarked == "C").astype(int)
FEATURES = ["Pclass", "FamilySize", "TicketCount", "FarePerPerson", "AgeFilled", "FromS", "FromC"]

cv = StratifiedKFold(5, shuffle=True, random_state=0)
models = {
    "Random forest": RandomForestClassifier(300, max_depth=4, min_samples_leaf=5, random_state=0),
    "Gradient boosting": GradientBoostingClassifier(n_estimators=100, max_depth=2, random_state=0),
    "Logistic regression": LogisticRegression(max_iter=2000),
}
segments = {
    "Adult men": ~df.IsWomanOrChild,
    "Women, no group": (df.Sex == "female") & (df.GroupId == "none"),
    "3rd class women, no group": (df.Sex == "female") & (df.GroupId == "none") & (df.Pclass == 3),
}
rows = []
for seg, mask in segments.items():
    part = df[is_train & mask]
    y = part.Survived.astype(int)
    row = {"Segment": seg, "Passengers": len(part),
           "Simple guess": (y == (part.Sex == "female")).mean()}
    for name, m in models.items():
        row[name] = cross_val_score(m, part[FEATURES], y, cv=cv).mean()
    rows.append(row)
pd.DataFrame(rows).set_index("Segment").round(3)

,Passengers,Simple guess,Random forest,Gradient boosting,Logistic regression
Segment,,,,,
Adult men,537,0.840,0.836,0.832,0.836
"Women, no group",198,0.818,0.783,0.778,0.777
"3rd class women, no group",77,0.623,0.597,0.572,0.597


**Every model is at or below the simple guess.** So the models make things worse.

### Last check: only act when the model is very sure
Maybe the model is right when it's confident. Count how many passengers it would flip, and how many flips are right.

In [6]:
for seg in ["Adult men", "Women, no group"]:
    part = df[is_train & segments[seg]]
    y = part.Survived.astype(int)
    prob = cross_val_predict(models["Random forest"], part[FEATURES], y, cv=cv, method="predict_proba")[:, 1]
    for cut in [0.5, 0.6, 0.7]:
        if seg == "Adult men":
            flip = prob >= cut            # predict a man lives
            right = (y[flip] == 1).sum()
        else:
            flip = prob <= 1 - cut        # predict a woman dies
            right = (y[flip] == 0).sum()
        print(f"{seg:<16} sure > {cut:.0%}: flips {flip.sum():>3}, right {right:>3}, wrong {flip.sum() - right:>3}")

Adult men        sure > 50%: flips   8, right   3, wrong   5
Adult men        sure > 60%: flips   0, right   0, wrong   0
Adult men        sure > 70%: flips   0, right   0, wrong   0


Women, no group  sure > 50%: flips   9, right   1, wrong   8
Women, no group  sure > 60%: flips   0, right   0, wrong   0
Women, no group  sure > 70%: flips   0, right   0, wrong   0


The model is **never** more than 60% sure, and at 50% it gets more flips wrong than right.
The data has no clear signal for these passengers. Their survival was mostly luck, so **idea 2 is dropped.**

## Write the submission

In [7]:
sub = pd.DataFrame({"PassengerId": tst.PassengerId.astype(int), "Survived": pred.astype(int).values})
sub.to_csv("submission_v3.csv", index=False)
print(len(sub), "rows,", f"{sub.Survived.mean():.0%} predicted survived")

418 rows, 36% predicted survived


## Summary
- **Ticket groups:** changes 1 passenger. Expect the same score or about +0.2% (1 of 418 is 0.24%).
- **Models for the rest:** tried 3 models on 3 groups of passengers. None beat the simple guess.
- **Takeaway:** about 80% is close to the limit for honest approaches on this data.
  Submitting many small changes and keeping the best leaderboard number is tuning to luck, not real progress.